# Spike sorting on a single IT electrode

A 2,390-second extracellular recording from macaque inferotemporal cortex,
sampled at 30 kHz (71,698,898 samples), taken from raw voltage to sorted
units: band-pass filter, amplitude threshold at a robust noise estimate,
waveform extraction, PCA, and k-means over the principal components.

The notebook also does something the original did not: it checks whether the
accompanying "ground truth" file can validate any of it.

## The headline

`Spikes (1).mat` does not contain spike times. Its single array,
`ind_spikes_it`, is a strictly increasing list of 90,789 integers spanning
1 to 138,879, of which **73.7% of consecutive differences are exactly 1**.
A spike train cannot look like that — at 1 kHz it would mean 67,000 spikes
separated by one millisecond, through a refractory period. It is a *selection
index* into an event list of 138,879 entries that is not in the file.

The original comparison intersected detected sample indices with those
ordinals and reported the result as a detection score (F1 = 0.018). Section 3
reproduces that number and section 4 shows where it comes from: every one of
the 1,004 "matches" falls below 138,879, i.e. inside the first 138.9 s of a
2,390 s recording, and inside that range the hit rate (72.5%) is what you would
get by drawing integers at random (the index covers 65.4% of its own range).
The comparison measured integer collisions, not detection quality.

Section 6 replaces it with validation that needs no ground truth: silhouette
separation in PCA space and refractory-period violations per cluster.

## Defects repaired here

| # | Defect | Consequence |
|---|---|---|
| 1 | `find_peaks` called on `+x` only | Extracellular spikes are negative-going. Detection found 8,999 events where both polarities give 31,064 — **71% of the events were never seen** |
| 2 | Detected indices at `fs/ds` compared against a 1 kHz index | Only the `DS=30` row of the decimation sweep compared like with like; the other eight rows were meaningless even on their own terms |
| 3 | `np.intersect1d` with no tolerance | Requires a sample-exact hit, so even a correct detector would score near zero |
| 4 | The comparison itself | See above — the reference array is not a spike train |

Fixes are marked `FIX` at the site. Metrics quoted in the README come from a
re-run of this notebook.

In [ ]:
import os
from pathlib import Path

# The recordings are hundreds of megabytes and are not committed. Point
# SPIKE_DATA_DIR at wherever they live, or drop them in ./data/.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = Path(os.environ.get('SPIKE_DATA_DIR') or ROOT / 'data')

if not DATA.is_dir():
    raise SystemExit(
        'No data directory at %s.\n'
        'Set SPIKE_DATA_DIR or create ./data/ — see the README.' % DATA)
print('data:', DATA)


## 1. Detection and clustering

High-pass at 300 Hz to remove the local field potential, then threshold at
5σ where σ is the robust noise estimate

$$\sigma_n = \mathrm{median}\left(\frac{|x|}{0.6745}\right)$$

which is the median absolute deviation rescaled to the standard deviation of a
Gaussian. The point of using the median is that it is barely moved by the
spikes themselves, so the threshold does not rise with the firing rate the way
a plain standard deviation would.

Waveforms are cut ±2 ms around each peak, reduced to three principal
components, and clustered with k-means for k = 2…5.

In [ ]:
import h5py, numpy as np, matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
MAT_FILE      = DATA / 'singleIT.mat'
DATASET       = 'data_IT'
FS            = 30_000
SCALE_FACTOR  = 10
HP_CUTOFF     = 300
HP_ORDER      = 7
THR_MULT      = 5
WIN_MS        = 2
K_RANGE       = range(2, 6)
with h5py.File(MAT_FILE, 'r') as f:
    if DATASET not in f:
        raise KeyError(f"'{DATASET}' not found in {MAT_FILE}")
    data = np.array(f[DATASET]).flatten()
data = data / SCALE_FACTOR
n_samples = data.size
time_s = np.arange(n_samples) / FS
print(f'data length = {n_samples:,} samples  ({n_samples/FS:.1f} s)')
plt.figure(figsize=(6,4))
plt.hist(data, bins=200, color='steelblue', alpha=.8)
plt.title('Voltage histogram (µV)')
plt.xlabel('Voltage (µV)'); plt.ylabel('Count')
plt.tight_layout(); plt.show()
def butter_hp(cut, fs, order):
    b, a = butter(order, cut/(0.5*fs), 'high')
    return b, a
b, a = butter_hp(HP_CUTOFF, FS, HP_ORDER)
data_filt = filtfilt(b, a, data)
plt.figure(figsize=(14,5))
plt.subplot(2,1,1)
plt.plot(time_s, data, lw=.3)
plt.title('Raw signal'); plt.ylabel('µV')
plt.subplot(2,1,2)
plt.plot(time_s, data_filt, lw=.3)
plt.title(f'High-pass {HP_CUTOFF} Hz (order {HP_ORDER})')
plt.xlabel('Time (s)'); plt.ylabel('µV')
plt.tight_layout(); plt.show()
sigma_n = np.median(np.abs(data_filt)/0.6745)
theta   = THR_MULT * sigma_n
print(f'σₙ = {sigma_n:.3f} µV   →   θ = {theta:.3f} µV')
# FIX 1. The original was `find_peaks(data_filt, ...)` alone, which
# keeps positive peaks only. Extracellular action potentials are
# predominantly negative-going, and on this recording the two polarities
# give 8,999 against 22,065 events.
_dist = int(0.001*FS)
pos, _ = find_peaks( data_filt, height=theta, distance=_dist)
neg, _ = find_peaks(-data_filt, height=theta, distance=_dist)
peaks  = np.sort(np.concatenate([pos, neg]))
print(f'spikes detected: {len(peaks)}  (positive {pos.size}, negative {neg.size})')
if len(peaks) == 0:
    raise RuntimeError('No spikes above threshold – adjust θ or check data.')
w      = int(WIN_MS*FS/1000)
t_ms   = np.linspace(-WIN_MS, WIN_MS, 2*w)
waves  = []
for p in peaks:
    if p-w >= 0 and p+w < n_samples:
        waves.append(data_filt[p-w:p+w])
waveforms = np.vstack(waves)
print('waveforms matrix:', waveforms.shape)
plt.figure(figsize=(8,4))
plt.plot(t_ms, waveforms.T, color='gray', alpha=.25)
plt.title('All spike waveforms (±2 ms)')
plt.xlabel('Time (ms)'); plt.ylabel('µV')
plt.tight_layout(); plt.show()
pca = PCA(n_components=3)
pcs = pca.fit_transform(waveforms)
print('explained variance ratio:', pca.explained_variance_ratio_)
plt.figure(figsize=(5,4))
plt.scatter(pcs[:,0], pcs[:,1], s=5, alpha=.4)
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('PC1 vs PC2')
plt.grid(True); plt.tight_layout(); plt.show()
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=0)
    labels = km.fit_predict(pcs)
    fig, (ax1, ax2) = plt.subplots(1,2,figsize=(9,4))
    for lab in range(k):
        ax1.scatter(pcs[labels==lab,0], pcs[labels==lab,1], s=5, label=f'C{lab}')
        ax2.scatter(pcs[labels==lab,0], pcs[labels==lab,2], s=5)
    ax1.set_xlabel('PC1'); ax1.set_ylabel('PC2')
    ax1.set_title(f'k={k}: PC1 vs PC2'); ax1.legend(markerscale=3, fontsize=8)
    ax2.set_xlabel('PC1'); ax2.set_ylabel('PC3')
    ax2.set_title('PC1 vs PC3')
    plt.tight_layout(); plt.show()
tsne = TSNE(n_components=2, perplexity=30, random_state=0)
pcs_tsne = tsne.fit_transform(pcs)
plt.figure(figsize=(6,5))
plt.scatter(pcs_tsne[:,0], pcs_tsne[:,1], s=5, alpha=0.6)
plt.title('t-SNE visualization of spike waveforms (on PCA components)')
plt.xlabel('t-SNE dim 1')
plt.ylabel('t-SNE dim 2')
plt.grid(True)
plt.tight_layout()
plt.show()
for k in K_RANGE:
    km_tsne = KMeans(n_clusters=k, n_init=10, random_state=0)
    labels_tsne = km_tsne.fit_predict(pcs_tsne)
    plt.figure(figsize=(6,5))
    for lab in range(k):
        plt.scatter(pcs_tsne[labels_tsne==lab,0], pcs_tsne[labels_tsne==lab,1],
                    s=5, label=f'Cluster {lab}', alpha=0.7)
    plt.title(f't-SNE + K-means clustering (k={k})')
    plt.xlabel('t-SNE dim 1')
    plt.ylabel('t-SNE dim 2')
    plt.legend(markerscale=3, fontsize=8)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

sorting = dict(peaks=peaks, waveforms=waveforms, sigma_n=sigma_n,
               fs=FS, win_ms=WIN_MS, n_samples=n_samples)


t-SNE run directly on the 120-sample waveforms rather than on the principal
components. PCA first is the usual choice — it denoises and makes t-SNE much
cheaper — so this is a check that the three components are not discarding
structure that separates units.

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=0)
waveforms_tsne = tsne.fit_transform(waveforms)
plt.figure(figsize=(6,5))
plt.scatter(waveforms_tsne[:,0], waveforms_tsne[:,1], s=5, alpha=0.7)
plt.title('t-SNE visualization of spike waveforms (direct)')
plt.xlabel('t-SNE dim 1')
plt.ylabel('t-SNE dim 2')
plt.grid(True)
plt.tight_layout()
plt.show()


## 2. What is actually in the ground-truth file

Before comparing anything against `Spikes (1).mat`, look at it.

In [ ]:
import h5py
import numpy as np

with h5py.File(DATA / 'Spikes (1).mat', 'r') as f:
    keys = [k for k in f if isinstance(f[k], h5py.Dataset)]
    print('datasets:', {k: f[k].shape for k in keys})
    truth = np.array(f['ind_spikes_it']).flatten().astype(np.int64)

gaps = np.diff(truth)
print('n = %d   range = %d .. %d' % (truth.size, truth.min(), truth.max()))
print('strictly increasing:', bool((gaps > 0).all()))
print('fraction of gaps equal to 1: %.3f' % (gaps == 1).mean())
print('covers %.3f of its own range' % (truth.size / truth.max()))
print('last five:', truth[-5:])

# A spike train sampled at 1 kHz over a 2,390 s recording would span ~2.39e6,
# not 1.4e5, and its inter-spike gaps could not be one sample 74% of the time
# — that is shorter than any refractory period. This is an index into an
# event list of 138,879 entries, not a list of times.


## 3. Reproducing the original comparison

The original swept nine decimation factors and counted exact index
intersections against `ind_spikes_it`. It is reproduced here unchanged apart
from the path and the Persian comments, because its output is the evidence the
next section explains.

Two things are wrong with it before the ground truth is even considered.
Detected indices are in units of `fs/ds`, while the reference array is not —
so the rows are not on a common axis (**defect 2**); and `np.intersect1d`
demands a sample-exact hit, which no real detector achieves (**defect 3**).

In [ ]:
"""Detection against the reference array, as originally written.

- band-pass 300-3000 Hz
- sweep of decimation factors
- positive and negative peaks at a sigma-multiplier threshold
- exact index intersection, no unit conversion

Kept for reproduction only. Sections 2 and 4 show why the counts it prints
do not measure detection quality.
"""
from scipy.signal import butter, filtfilt, find_peaks
MAT_RAW     = "singleIT.mat"      # resolved against DATA
DATA_KEY    = "data_IT"
MAT_TRUTH   = "Spikes (1).mat"   # resolved against DATA
TRUTH_CANDS = ["spikes","spikeTimes","trueSpikes","truth"]
FS          = 30000.0
LOWCUT      = 300.0
HIGHCUT     = 3000.0
FILTER_ORDER= 4
MIN_DIST_MS = 1.0
THR_MULT    = 4.0
DS_FACTORS  = [1, 2, 3, 5, 10, 20, 30, 60, 100]
def load_raw(fname, key):
    with h5py.File(fname, "r") as f:
        data = f[key][()]
    return np.array(data).flatten()
def load_truth(fname, candidates):
    with h5py.File(fname, "r") as f:
        for k in candidates:
            if k in f and isinstance(f[k], h5py.Dataset):
                d = f[k][()]
                if d.ndim==1 or (d.ndim==2 and 1 in d.shape):
                    return np.array(d).flatten().astype(int)
        best, best_len = None, 0
        def visitor(name, obj):
            nonlocal best, best_len
            if isinstance(obj, h5py.Dataset):
                shp = obj.shape
                if obj.ndim==1 or (obj.ndim==2 and 1 in shp):
                    length = shp[0] if obj.ndim==1 else (shp[0] if shp[1]==1 else shp[1])
                    if length>best_len:
                        best, best_len = obj, length
        f.visititems(visitor)
        if best is not None:
            return np.array(best).flatten().astype(int)
    raise RuntimeError("No 1D dataset found in truth file")
def bandpass(x, fs, low, high, order=4):
    b, a = butter(order, [low, high], btype='band', fs=fs)
    return filtfilt(b, a, x)
def detect_spikes(sig, fs, min_dist_ms, thr_mult):
    sigma_n = np.median(np.abs(sig)) / 0.6745
    thr = thr_mult * sigma_n
    min_dist = max(1, int(min_dist_ms * fs / 1000))
    p_pos, _ = find_peaks(sig,    height=thr, distance=min_dist)
    p_neg, _ = find_peaks(-sig,   height=thr, distance=min_dist)
    return np.sort(np.concatenate([p_pos, p_neg]))
if __name__=="__main__":
    raw = load_raw(DATA / MAT_RAW, DATA_KEY)
    print(f"Loaded raw: {raw.size} samples @ {FS:.0f} Hz")
    raw_f = bandpass(raw, FS, LOWCUT, HIGHCUT, FILTER_ORDER)
    truth = load_truth(DATA / MAT_TRUTH, TRUTH_CANDS)
    print(f"Loaded ground-truth: {truth.size} indices")
    results = []
    for ds in DS_FACTORS:
        fs_ds = FS / ds
        sig_ds = raw_f[::ds]
        det = detect_spikes(sig_ds, fs_ds, MIN_DIST_MS, THR_MULT)
        # DEFECT 2. `det` is in units of fs/ds; `truth` is not in units of
        # anything comparable. Only DS=30 puts the two on the same nominal
        # axis, which is why that row stands out below.
        # DEFECT 3. Exact intersection — a sample-exact hit is required.
        hits = np.intersect1d(det, truth, assume_unique=True).size
        results.append((ds, truth.size, det.size, hits))
        print(f"DS={ds:3d} → truth={truth.size:6d}, detected={det.size:6d}, matched={hits:6d}")
    print("\nSummary:")
    for ds, Ntruth, Ndet, Nhits in results:
        print(f"  DS={ds:3d}: matched {Nhits}/{Ntruth} ({Nhits/Ntruth*100:.2f} %)")


## 4. Where the "matches" come from

`DS=30` reported the most matches, 1,004, and was read as the best
configuration. This section takes that row apart.

In [ ]:
# The DS=30 row, re-derived so the numbers can be interrogated.
sig_ds = bandpass(load_raw(DATA / MAT_RAW, DATA_KEY), FS, LOWCUT, HIGHCUT,
                  FILTER_ORDER)[::30]
det = detect_spikes(sig_ds, FS / 30, MIN_DIST_MS, THR_MULT)
hits = np.intersect1d(det, truth)

in_range = int((det < truth.max()).sum())
print('detected           %6d' % det.size)
print('matched            %6d' % hits.size)
print('largest match      %6d   (reference array ends at %d)'
      % (hits.max(), truth.max()))
print('largest detection  %6d' % det.max())
print()
print('Every match lies below %d — the first %.1f s of a %.1f s recording.'
      % (truth.max(), truth.max() / (FS / 30), sig_ds.size / (FS / 30)))
print('Detections in that range:            %5d' % in_range)
print('Of those, "matched":                 %5d  (%.1f%%)'
      % (hits.size, 100 * hits.size / in_range))
print('Base rate — how much of 1..%d the reference covers: %.1f%%'
      % (truth.max(), 100 * truth.size / truth.max()))
print()
print('A detector with no relationship to the reference scores the base rate.')
print('This one scores the base rate. Nothing was measured.')


## 5. Threshold choice at 1 kHz

Two thresholds on the decimated signal: the robust 4σ estimate, and
0.9·max|x|. The second is included because the report used it, and it is worth
keeping as a demonstration — taking 90% of the single largest excursion in
2,390 s of recording puts the threshold above every spike but one, so it
detects exactly one event. Precision, recall and F1 are all reported as 0.000.

The precision/recall figures in this cell inherit the problem from section 4
and mean nothing. The PCA and t-SNE embeddings of the *detected* waveforms are
unaffected and are the useful output; the `matched` embeddings are shown for
contrast, and are what a coincidence-selected subsample looks like.

In [ ]:
"""Threshold comparison on the decimated signal.

- band-pass 300-3000 Hz (order 4), decimate 30 kHz -> 1 kHz
- three thresholds: 4*sigma_n, 8*sigma_n and 0.9*max|x|
- PCA and t-SNE on the detected waveforms

Two caveats, both inherited from the original and both left visible rather
than papered over:

1. The precision/recall block is kept because the report quotes it, but see
   section 4 — the reference array is not a spike train, so those three
   numbers describe integer collisions. The embeddings are unaffected.
2. `0.9 * max|x|` behaves as described in DEFECT 4 below.
"""
MAT_RAW     = DATA / "singleIT.mat"
KEY_RAW     = "data_IT"
MAT_GT      = DATA / "Spikes (1).mat"
FS_RAW      = 30000.0
DS          = 30
FS_DS       = FS_RAW/DS
LOWCUT, HIGHCUT = 300.0, 3000.0
ORDER       = 4
THR_MULT    = 4.0
WIN_MS      = 2.0
MIN_DIST_MS = 1.0
def load_raw():
    with h5py.File(MAT_RAW,'r') as f:
        return f[KEY_RAW][()].flatten()
def load_truth():
    best, best_len, name = None, 0, None
    with h5py.File(MAT_GT,'r') as f:
        def visitor(n,obj):
            nonlocal best, best_len, name
            if isinstance(obj,h5py.Dataset):
                d = np.array(obj[()]).flatten()
                if d.ndim==1 and d.size>best_len:
                    best_len = d.size; best = d.astype(int); name = n
        f.visititems(visitor)
    if best is None:
        raise RuntimeError("No 1-D dataset in ground-truth file")
    print(f"Using GT dataset '{name}' with {best_len} events")
    return best
def bandpass(x, fs):
    b,a = butter(ORDER, [LOWCUT, HIGHCUT], btype='band', fs=fs)
    return filtfilt(b, a, x)
def detect(sig, fs, thr):
    d = max(1, int(MIN_DIST_MS*fs/1000))
    p,_ = find_peaks(sig, height=thr, distance=d)
    n,_ = find_peaks(-sig, height=thr, distance=d)
    return np.sort(np.concatenate([p,n]))
def extract_waveforms(sig, peaks, fs, win_ms):
    w = int(win_ms*fs/1000)
    waves=[]
    for p in peaks:
        if p-w>=0 and p+w<sig.size:
            waves.append(sig[p-w:p+w])
    return np.array(waves)
def plot_feats(feats, title):
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(10,4))
    ax1.scatter(feats[:,0],feats[:,1],s=5,alpha=.5); ax1.set_xlabel('C1'); ax1.set_ylabel('C2')
    ax2.scatter(feats[:,0],feats[:,2],s=5,alpha=.5); ax2.set_xlabel('C1'); ax2.set_ylabel('C3')
    fig.suptitle(title); plt.tight_layout(); plt.show()
raw = load_raw()
sig_f = bandpass(raw, FS_RAW)
sig_ds = sig_f[::DS]
truth = load_truth()
sigma_n   = np.median(np.abs(sig_ds)) / 0.6745
thr_sigma = THR_MULT * sigma_n
thr_max   = 0.9 * np.max(np.abs(sig_ds))
print(f"θσ={thr_sigma:.3f}, θmax={thr_max:.3f}, GT spikes={truth.size}")
# DEFECT 4. `0.9 * max|x|` is not a threshold choice, it is a way of
# selecting the single largest sample in the recording: nothing short of
# the extremum clears 90 % of it, and on this data it detects exactly one
# event — enough to crash PCA, which is how it was found. The arm is kept
# because the report quotes its F1, but the comparison the section is
# reaching for is liberal-vs-conservative in units of the noise, so
# 8*sigma_n is added alongside it.
dets = {
    '4σ':        detect(sig_ds, FS_DS, thr_sigma),
    '8σ':        detect(sig_ds, FS_DS, 8.0 * sigma_n),
    '0.9·max|x|': detect(sig_ds, FS_DS, thr_max),
}
for tag, peaks in dets.items():
    TP = np.intersect1d(peaks, truth).size
    FP = peaks.size - TP
    FN = truth.size - TP
    prec = TP/(TP+FP) if TP+FP>0 else 0
    rec  = TP/(TP+FN) if TP+FN>0 else 0
    f1   = 2*prec*rec/(prec+rec) if prec+rec>0 else 0
    print(f"\n{tag}: det={peaks.size}, TP={TP}, FP={FP}, FN={FN}")
    print(f"Prec={prec:.3f}, Rec={rec:.3f}, F1={f1:.3f}")
    for mode, idxs in [('all', peaks), ('matched', np.intersect1d(peaks, truth))]:
        wav = extract_waveforms(sig_ds, idxs, FS_DS, WIN_MS)
        print(f"{tag}-{mode}: {wav.shape[0]} waveforms")
        if wav.shape[0] < 50:
            # t-SNE needs more samples than its perplexity, and PCA needs
            # more than n_components. Below this the embedding is noise.
            print("  too few waveforms to embed — skipped")
            continue
        pcs = PCA(n_components=3).fit_transform(wav)
        plot_feats(pcs, f"{tag}-{mode} PCA")
        emb = TSNE(n_components=3, init='pca', learning_rate='auto',
                   random_state=0, perplexity=30).fit_transform(wav)
        plot_feats(emb, f"{tag}-{mode} t-SNE")


## 6. Validation without ground truth

With no reference spike train, detection quality has to be judged from the
data itself. Two standard measures:

**Silhouette score** in PCA space — how much better each waveform matches its
own cluster than the nearest other one. It says whether the clusters are
separated at all, without reference to what they mean.

**Refractory-period violations** — a real neuron cannot fire twice within
about 2 ms. The fraction of within-cluster inter-spike intervals below 2 ms is
therefore a contamination estimate: a well-isolated single unit sits near zero,
a few percent indicates a multi-unit cluster, and a cluster in the tens of
percent is not a neuron at all.

The 1 ms `distance` argument in `find_peaks` makes violations below 1 ms
impossible by construction, so this measures the 1–2 ms band only. It still
separates units from artifacts, which is what it is used for here.

In [ ]:
from sklearn.metrics import silhouette_score

def cluster_quality(peak_idx, waves, fs, sigma, k_range=(2, 3, 4), isi_ms=2.0):
    """Silhouette separation and refractory violations, per k."""
    pcs = PCA(n_components=3).fit_transform(waves)
    for k in k_range:
        labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(pcs)
        parts = []
        for c in range(k):
            t_ms = np.sort(peak_idx[labels == c]) / fs * 1000.0
            isi = np.diff(t_ms)
            frac = (isi < isi_ms).mean() * 100 if isi.size else float('nan')
            parts.append('C%d n=%-5d ISI<%gms %5.1f%%' % (c, t_ms.size, isi_ms, frac))
        print('k=%d  silhouette %.3f | %s'
              % (k, silhouette_score(pcs, labels), '  '.join(parts)))
    print('SNR (median |peak| / sigma_n) = %.2f'
          % (np.median(np.abs(waves).max(axis=1)) / sigma))

# Section 1 drops any peak too close to an edge to cut a full window, so the
# peak list and the waveform matrix have to be realigned before pairing them.
w_pts = int(sorting['win_ms'] * sorting['fs'] / 1000)
edge = ((sorting['peaks'] - w_pts >= 0)
        & (sorting['peaks'] + w_pts < sorting['n_samples']))
kept = sorting['peaks'][edge]
assert kept.size == sorting['waveforms'].shape[0]

cluster_quality(kept, sorting['waveforms'], sorting['fs'], sorting['sigma_n'])
